### Determinação de parâmetros das redes — tamanho de janela \(L_M\)

Análise inspirada em Rosário (2017): número médio de arestas \(N_a\) em função do tamanho da janela \(L_M\) (em amostras),
comparando redes **originais** e **embaralhadas** (motifs com eixo temporal aleatorizado).

Objetivo: identificar o menor \(L_M\) a partir do qual a rede original se distancia da nula.

Fluxo:
1. Carrega 1 EEG em `data/preprocessed`
2. Motifs simbólicos
3. Para cada \(L_M\): threshold (surrogate) → TVG original e TVG embaralhado → \(N_a\)
4. Gráfico \(N_a\) vs \(L_M\) + interpretação (e equivalente em ms)

### Imports

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import mne
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Cursor/VS Code: evita erro de nbformat em sessões antigas do kernel
pio.renderers.default = "vscode"
try:
    import nbformat as _nbformat
    import plotly.io._renderers as _plotly_renderers

    _plotly_renderers.nbformat = _nbformat
except Exception:
    pass

from src.graph.aggregate import (
    aggregate_static_network,
    edges_per_time,
    temporal_statistics,
)
from src.graph.motifs import trans_motifs
from src.graph.network import graph_metrics
from src.graph.threshold import estimate_threshold
from src.graph.tvg import build_tvg_parallel


### Paths

In [2]:
SOURCE_DIR = Path("../../data/preprocessed")

print("SOURCE_DIR:", SOURCE_DIR.resolve())

SOURCE_DIR: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\preprocessed


### Parâmetros

In [3]:
N_SEC = 5
WINDOW_SIZES = list(range(5, 26))  # L_M em amostras (como na figura de Rosário)
WINDOW_SIZES_ML = [8, 10, 12, 15, 20, 25]  # grade reduzida para downstream
MAX_LAG_CAP = 5  # lag máximo em amostras; por janela: min(MAX_LAG_CAP, L_M - 1)
N_SURROGATES = 100
THRESHOLD_PERCENTILE = 95.0
RANDOM_STATE = 42
N_SUBJECTS_ML = 3  # pares EO/EC no método downstream


### Listar arquivos e escolher 1 EEG

In [4]:
files = sorted(SOURCE_DIR.glob("sub-*.set"))
print(f"Encontrados {len(files)} arquivos")
files[:5]

Encontrados 315 arquivos


[WindowsPath('../../data/preprocessed/sub-010060_EO.set'),
 WindowsPath('../../data/preprocessed/sub-010061_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010061_EO.set'),
 WindowsPath('../../data/preprocessed/sub-010062_EC.set'),
 WindowsPath('../../data/preprocessed/sub-010062_EO.set')]

In [5]:
file = files[0]
file

WindowsPath('../../data/preprocessed/sub-010060_EO.set')

### Carregar EEG e motifs

In [6]:
# ==========================
# LOAD EEG
# ==========================
raw = mne.io.read_raw_eeglab(file, preload=True, verbose=False)
raw.pick_types(eeg=True)

eeg = raw.get_data()
sfreq = float(raw.info["sfreq"])
eeg = eeg[:, : int(N_SEC * sfreq)]

print(f"Arquivo: {file.name}")
print(f"Shape EEG: {eeg.shape} | Fs: {sfreq} Hz | Duração: {N_SEC} s")

# ==========================
# MOTIFS
# ==========================
motifs = trans_motifs(eeg)
print(f"Shape motifs: {motifs.shape}")

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Arquivo: sub-010060_EO.set
Shape EEG: (61, 1250) | Fs: 250.0 Hz | Duração: 5 s
Shape motifs: (61, 1248)


C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\323120992.py:4: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\323120992.py:4: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.



### Varredura em \(L_M\): original vs embaralhado

Para cada tamanho de janela: estima threshold, constrói TVG original e TVG com motifs embaralhados no tempo (mesmo threshold), e calcula \(N_a\) médio.

**Atenção:** cada \(L_M\) roda 2 TVGs — pode demorar vários minutos.

In [7]:
rng = np.random.default_rng(RANDOM_STATE)

rows = []

for L_M in tqdm(WINDOW_SIZES, desc="L_M"):
    max_lag = min(MAX_LAG_CAP, L_M - 1)

    # ==========================
    # THRESHOLD (surrogate)
    # ==========================
    threshold_info = estimate_threshold(
        motifs,
        L_M,
        max_lag,
        n_surrogates=N_SURROGATES,
        percentile=THRESHOLD_PERCENTILE,
        random_state=RANDOM_STATE,
    )
    threshold = threshold_info["threshold"]

    # ==========================
    # TVG ORIGINAL
    # ==========================
    tvg_orig = build_tvg_parallel(
        motifs=motifs,
        window_size=L_M,
        max_lag=max_lag,
        threshold=threshold,
    )
    na_orig = float(np.mean(edges_per_time(tvg_orig)))

    # ==========================
    # MOTIFS EMBARALHADOS + TVG
    # ==========================
    motifs_shuf = motifs.copy()
    for ch in range(motifs_shuf.shape[0]):
        rng.shuffle(motifs_shuf[ch])

    tvg_shuf = build_tvg_parallel(
        motifs=motifs_shuf,
        window_size=L_M,
        max_lag=max_lag,
        threshold=threshold,
    )
    na_shuf = float(np.mean(edges_per_time(tvg_shuf)))

    rows.append(
        {
            "L_M": L_M,
            "L_M_ms": L_M / sfreq * 1000,
            "max_lag": max_lag,
            "threshold": threshold,
            "Na_original": na_orig,
            "Na_embaralhado": na_shuf,
        }
    )
    print(
        f"L_M={L_M:2d} pts ({L_M / sfreq * 1000:.1f} ms) | "
        f"thr={threshold:.4f} | Na_orig={na_orig:.1f} | Na_shuf={na_shuf:.1f}"
    )

df_na = pd.DataFrame(rows)
df_na

L_M:   0%|          | 0/21 [00:00<?, ?it/s]

L_M:   5%|▍         | 1/21 [01:02<20:58, 62.92s/it]

L_M= 5 pts (20.0 ms) | thr=1.0000 | Na_orig=789.2 | Na_shuf=782.0


L_M:  10%|▉         | 2/21 [01:47<16:32, 52.21s/it]

L_M= 6 pts (24.0 ms) | thr=1.0000 | Na_orig=775.9 | Na_shuf=783.1


...

L_M:  95%|█████████▌| 20/21 [15:37<00:34, 34.15s/it]

L_M=24 pts (96.0 ms) | thr=0.5789 | Na_orig=855.4 | Na_shuf=143.2


L_M: 100%|██████████| 21/21 [16:06<00:00, 46.04s/it]

L_M=25 pts (100.0 ms) | thr=0.5909 | Na_orig=786.2 | Na_shuf=97.5


,L_M,L_M_ms,max_lag,threshold,Na_original,Na_embaralhado
0,5,20.0,4,1.000000,789.179984,781.963680
1,6,24.0,5,1.000000,775.902991,783.136621
2,7,28.0,5,1.000000,555.615696,291.018608
3,8,32.0,5,1.000000,400.199190,104.497166
4,9,36.0,5,1.000000,292.362237,36.385737
5,10,40.0,5,0.889444,289.102190,13.164639
6,11,44.0,5,0.834524,318.263799,26.039773
7,12,48.0,5,0.820130,379.918765,24.019496
8,13,52.0,5,0.694056,591.990244,119.655285
9,14,56.0,5,0.642857,827.593979,212.227828


### Gráfico \(N_a\) vs \(L_M\)

In [8]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_na["L_M"],
        y=df_na["Na_original"],
        mode="lines+markers",
        name="Redes originais",
        marker=dict(symbol="square", size=8, color="black"),
        line=dict(color="black", width=2),
    )
)

fig.add_trace(
    go.Scatter(
        x=df_na["L_M"],
        y=df_na["Na_embaralhado"],
        mode="lines+markers",
        name="Redes embaralhadas",
        marker=dict(symbol="circle", size=8, color="crimson"),
        line=dict(color="crimson", width=2),
    )
)

fig.update_layout(
    title=(
        f"Número médio de arestas vs tamanho da janela — {file.stem}<br>"
        "<sup>Comparação redes originais vs embaralhadas (Rosário, 2017)</sup>"
    ),
    xaxis_title="Tamanho da janela L_M (pts)",
    yaxis_title="N_a (número médio de arestas)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(x=0.02, y=0.98),
    height=500,
)

display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

fig.add_trace(
    go.Scatter(
        x=df_na["L_M"],
        y=df_na["Na_embaralhado"],
        mode="lines+markers",
        name="Redes embaralhadas",
        marker=dict(symbol="circle", size=8, color="crimson"),
        line=dict(color="crimson", width=2),
    )
)

fig.update_layout(
    title=(
        f"Número médio de arestas vs tamanho da janela — {file.stem}<br>"
        "<sup>Comparação redes originais vs embaralhadas (Rosário, 2017)</sup>"
    ),
    xaxis_title="Tamanho da janela L_M (pts)",
    yaxis_title="N_a (número médio de arestas)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(x=0.02, y=0.98),
    height=500,
)
from IPython.display import HTML, display

display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


### Interpretação

Menor \(L_M\) em que a rede original se separa da embaralhada
(critério: \(N_a^{\text{orig}} > 2 \cdot N_a^{\text{shuf}}\) e diferença absoluta \(\geq 10\) arestas).

In [9]:
sep = (
    (df_na["Na_original"] > 2 * df_na["Na_embaralhado"])
    & ((df_na["Na_original"] - df_na["Na_embaralhado"]) >= 10)
)

if sep.any():
    row = df_na.loc[sep].iloc[0]
    L_star = int(row["L_M"])
    L_star_ms = float(row["L_M_ms"])
    print(f"Menor L_M com separação clara: {L_star} pts (~{L_star_ms:.1f} ms a {sfreq:.0f} Hz)")
    print(
        f"Na_original={row['Na_original']:.1f} | "
        f"Na_embaralhado={row['Na_embaralhado']:.1f}"
    )
    print(
        f"\nPara o pipeline (WINDOW_MS), use aproximadamente: "
        f"WINDOW_MS = {int(round(L_star_ms))} ms"
    )
else:
    print(
        "Nenhum L_M atendeu o critério de separação nesta faixa. "
        "Inspecione o gráfico ou ajuste N_SEC / surrogates / critério."
    )

df_na[["L_M", "L_M_ms", "Na_original", "Na_embaralhado"]]

Menor L_M com separação clara: 8 pts (~32.0 ms a 250 Hz)
Na_original=400.2 | Na_embaralhado=104.5

Para o pipeline (WINDOW_MS), use aproximadamente: WINDOW_MS = 32 ms


,L_M,L_M_ms,Na_original,Na_embaralhado
0,5,20.0,789.179984,781.963680
1,6,24.0,775.902991,783.136621
2,7,28.0,555.615696,291.018608
3,8,32.0,400.199190,104.497166
4,9,36.0,292.362237,36.385737
5,10,40.0,289.102190,13.164639
6,11,44.0,318.263799,26.039773
7,12,48.0,379.918765,24.019496
8,13,52.0,591.990244,119.655285
9,14,56.0,827.593979,212.227828


### Método 2 — Estabilidade temporal

Correlação média entre matrizes de adjacência **consecutivas** do TVG, para cada $L_M$.
Reutiliza o threshold já estimado em `df_na`.


In [10]:
# ==========================
# ESTABILIDADE TEMPORAL
# ==========================
thr_map = dict(zip(df_na["L_M"].astype(int), df_na["threshold"]))
rows_stab = []

for L_M in tqdm(WINDOW_SIZES, desc="Estabilidade"):
    max_lag = min(MAX_LAG_CAP, L_M - 1)
    threshold = float(thr_map.get(L_M, 0.0))

    tvg = build_tvg_parallel(
        motifs=motifs,
        window_size=L_M,
        max_lag=max_lag,
        threshold=threshold,
    )

    n_frames = tvg.shape[2]
    iu = np.triu_indices(tvg.shape[0], k=1)
    corrs = []
    for t in range(n_frames - 1):
        a = tvg[:, :, t][iu]
        b = tvg[:, :, t + 1][iu]
        if np.std(a) < 1e-12 or np.std(b) < 1e-12:
            corrs.append(0.0)
        else:
            corrs.append(float(np.corrcoef(a, b)[0, 1]))

    stability = float(np.nanmean(corrs)) if corrs else 0.0
    rows_stab.append({"L_M": L_M, "L_M_ms": L_M / sfreq * 1000, "stability": stability})
    print(f"L_M={L_M:2d} | stability={stability:.4f}")

df_stab = pd.DataFrame(rows_stab)
L_stab = int(df_stab.loc[df_stab["stability"].idxmax(), "L_M"])
print(f"\nL_M com maior estabilidade: {L_stab} pts (~{L_stab / sfreq * 1000:.1f} ms)")
df_stab


Estabilidade:   5%|▍         | 1/21 [00:12<04:01, 12.08s/it]

L_M= 5 | stability=0.5446


Estabilidade:  10%|▉         | 2/21 [00:25<04:06, 12.99s/it]

L_M= 6 | stability=0.5283


Estabilidade:  14%|█▍        | 3/21 [00:39<03:56, 13.14s/it]

...

Estabilidade:  95%|█████████▌| 20/21 [04:23<00:13, 13.00s/it]

L_M=24 | stability=0.9040


Estabilidade: 100%|██████████| 21/21 [04:36<00:00, 13.18s/it]

L_M=25 | stability=0.9094

L_M com maior estabilidade: 25 pts (~100.0 ms)


,L_M,L_M_ms,stability
0,5,20.0,0.544551
1,6,24.0,0.528280
2,7,28.0,0.609710
3,8,32.0,0.658094
4,9,36.0,0.686335
5,10,40.0,0.740390
6,11,44.0,0.776668
7,12,48.0,0.806154
8,13,52.0,0.831469
9,14,56.0,0.846953


In [11]:
fig = go.Figure(
    go.Scatter(
        x=df_stab["L_M"],
        y=df_stab["stability"],
        mode="lines+markers",
        name="Estabilidade temporal",
        marker=dict(size=8),
        line=dict(width=2),
    )
)
fig.update_layout(
    title="Estabilidade temporal vs L_M (correlação entre janelas consecutivas)",
    xaxis_title="Tamanho da janela L_M (pts)",
    yaxis_title="Correlação média entre frames consecutivos",
    template="plotly_white",
    hovermode="x unified",
    height=450,
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


### Método 3 — Split-half

Divide o trecho de motifs ao meio, constrói a REA de cada metade e correlaciona as matrizes (parte triangular superior).


In [12]:
# ==========================
# SPLIT-HALF
# ==========================
mid = motifs.shape[1] // 2
motifs_a = motifs[:, :mid]
motifs_b = motifs[:, mid:]
print(f"motifs_a: {motifs_a.shape} | motifs_b: {motifs_b.shape}")

rows_split = []
for L_M in tqdm(WINDOW_SIZES, desc="Split-half"):
    max_lag = min(MAX_LAG_CAP, L_M - 1)
    threshold = float(thr_map.get(L_M, 0.0))

    tvg_a = build_tvg_parallel(motifs_a, L_M, max_lag, threshold=threshold)
    tvg_b = build_tvg_parallel(motifs_b, L_M, max_lag, threshold=threshold)
    rea_a = aggregate_static_network(tvg_a)
    rea_b = aggregate_static_network(tvg_b)

    iu = np.triu_indices(rea_a.shape[0], k=1)
    va, vb = rea_a[iu], rea_b[iu]
    if np.std(va) < 1e-12 or np.std(vb) < 1e-12:
        corr = 0.0
    else:
        corr = float(np.corrcoef(va, vb)[0, 1])

    rows_split.append({"L_M": L_M, "L_M_ms": L_M / sfreq * 1000, "split_half_corr": corr})
    print(f"L_M={L_M:2d} | split-half r={corr:.4f}")

df_split = pd.DataFrame(rows_split)
L_split = int(df_split.loc[df_split["split_half_corr"].idxmax(), "L_M"])
print(f"\nL_M com maior correlação split-half: {L_split} pts (~{L_split / sfreq * 1000:.1f} ms)")
df_split


motifs_a: (61, 624) | motifs_b: (61, 624)


Split-half:   5%|▍         | 1/21 [00:11<03:46, 11.32s/it]

L_M= 5 | split-half r=0.6701


Split-half:  10%|▉         | 2/21 [00:24<03:52, 12.21s/it]

L_M= 6 | split-half r=0.5491


...

Split-half:  95%|█████████▌| 20/21 [04:14<00:12, 12.58s/it]

L_M=24 | split-half r=0.9107


Split-half: 100%|██████████| 21/21 [04:27<00:00, 12.72s/it]

L_M=25 | split-half r=0.9102

L_M com maior correlação split-half: 24 pts (~96.0 ms)


,L_M,L_M_ms,split_half_corr
0,5,20.0,0.670113
1,6,24.0,0.549111
2,7,28.0,0.623333
3,8,32.0,0.671901
4,9,36.0,0.697973
5,10,40.0,0.831196
6,11,44.0,0.796176
7,12,48.0,0.858440
8,13,52.0,0.857524
9,14,56.0,0.900736


In [13]:
fig = go.Figure(
    go.Scatter(
        x=df_split["L_M"],
        y=df_split["split_half_corr"],
        mode="lines+markers",
        name="Split-half",
        marker=dict(size=8, color="seagreen"),
        line=dict(width=2, color="seagreen"),
    )
)
fig.update_layout(
    title="Reprodutibilidade split-half vs L_M (correlação das REAs)",
    xaxis_title="Tamanho da janela L_M (pts)",
    yaxis_title="Correlação REA(1ª metade) vs REA(2ª metade)",
    template="plotly_white",
    hovermode="x unified",
    height=450,
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


### Método 4 — Critério fisiológico

Converte $L_M$ em milissegundos e compara com períodos típicos de bandas EEG (theta ~125 ms, alpha ~100 ms, beta ~50 ms). Não escolhe um único $L_M$; ancora a interpretação dos outros métodos.


In [14]:
# ==========================
# FISIOLÓGICO
# ==========================
BAND_PERIOD_MS = {
    "beta (~20 Hz)": 50.0,
    "alpha (~10 Hz)": 100.0,
    "theta (~8 Hz)": 125.0,
}

rows_phys = []
for L_M in WINDOW_SIZES:
    ms = L_M / sfreq * 1000
    if ms < 50:
        faixa = "curta demais (< ~1 ciclo beta)"
    elif ms <= 150:
        faixa = "compatível com alpha/beta (~50–150 ms)"
    else:
        faixa = "mais longa (> 1.5 ciclo alpha)"
    rows_phys.append(
        {
            "L_M": L_M,
            "L_M_ms": ms,
            "frac_alpha": ms / BAND_PERIOD_MS["alpha (~10 Hz)"],
            "frac_beta": ms / BAND_PERIOD_MS["beta (~20 Hz)"],
            "faixa": faixa,
        }
    )

df_phys = pd.DataFrame(rows_phys)
L_phys_min = int(df_phys.loc[df_phys["L_M_ms"] >= 50, "L_M"].min())
L_phys_max = int(df_phys.loc[df_phys["L_M_ms"] <= 150, "L_M"].max())
print(
    f"Faixa fisiológica sugerida: L_M = {L_phys_min}–{L_phys_max} pts "
    f"({L_phys_min / sfreq * 1000:.0f}–{L_phys_max / sfreq * 1000:.0f} ms a {sfreq:.0f} Hz)"
)
print("Referências de ciclo:", BAND_PERIOD_MS)
df_phys


Faixa fisiológica sugerida: L_M = 13–25 pts (52–100 ms a 250 Hz)
Referências de ciclo: {'beta (~20 Hz)': 50.0, 'alpha (~10 Hz)': 100.0, 'theta (~8 Hz)': 125.0}


,L_M,L_M_ms,frac_alpha,frac_beta,faixa
0,5,20.0,0.20,0.40,curta demais (< ~1 ciclo beta)
1,6,24.0,0.24,0.48,curta demais (< ~1 ciclo beta)
2,7,28.0,0.28,0.56,curta demais (< ~1 ciclo beta)
3,8,32.0,0.32,0.64,curta demais (< ~1 ciclo beta)
4,9,36.0,0.36,0.72,curta demais (< ~1 ciclo beta)
5,10,40.0,0.40,0.80,curta demais (< ~1 ciclo beta)
6,11,44.0,0.44,0.88,curta demais (< ~1 ciclo beta)
7,12,48.0,0.48,0.96,curta demais (< ~1 ciclo beta)
8,13,52.0,0.52,1.04,compatível com alpha/beta (~50–150 ms)
9,14,56.0,0.56,1.12,compatível com alpha/beta (~50–150 ms)


In [15]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_phys["L_M"],
        y=df_phys["L_M_ms"],
        mode="lines+markers",
        name="Duração da janela",
        marker=dict(size=8),
    )
)
for name, period in BAND_PERIOD_MS.items():
    fig.add_hline(y=period, line_dash="dash", annotation_text=name)

fig.update_layout(
    title="Duração de L_M (ms) vs períodos de bandas EEG",
    xaxis_title="Tamanho da janela L_M (pts)",
    yaxis_title="Duração (ms)",
    template="plotly_white",
    hovermode="x unified",
    height=450,
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


### Método 4b — IAF (pico alfa individual)

Estima a frequência alfa individual (IAF) em canais posteriores no trecho **EC** (olhos fechados),
antes do primeiro `boundary`. A janela sugerida corresponde a **~1 ciclo de alpha**:
$L_M = \mathrm{round}(f_s / f_{\alpha})$.


In [16]:
# ==========================
# IAF — escolher arquivo EC
# ==========================
# Preferir EC do mesmo sujeito do `file` atual; senão o primeiro EC da lista.
sid_atual = file.stem.split("_", 1)[0]
file_ec = None
for p in files:
    sid, cond = p.stem.split("_", 1)
    if sid == sid_atual and cond == "EC":
        file_ec = p
        break
if file_ec is None:
    for p in files:
        if p.stem.endswith("_EC"):
            file_ec = p
            break

if file_ec is None:
    raise FileNotFoundError("Nenhum arquivo *_EC.set encontrado em data/preprocessed.")

print(f"Arquivo EC para IAF: {file_ec.name}")


Arquivo EC para IAF: sub-010061_EC.set


In [17]:
# ==========================
# IAF — PSD Welch (8–12 Hz)
# ==========================
raw_ec = mne.io.read_raw_eeglab(file_ec, preload=True, verbose=False)
raw_ec.pick("eeg")

sfreq_iaf = float(raw_ec.info["sfreq"])

boundary_onsets = [
    ann["onset"]
    for ann in raw_ec.annotations
    if str(ann["description"]).lower() == "boundary"
]
tmax = min(boundary_onsets) if boundary_onsets else float(raw_ec.times[-1])
print(f"tmax usado: {tmax:.2f} s | boundaries: {len(boundary_onsets)}")

posteriores = [
    ch
    for ch in raw_ec.ch_names
    if ch.upper() in {"O1", "OZ", "O2", "PO3", "POZ", "PO4", "P3", "PZ", "P4"}
]
if not posteriores:
    posteriores = "eeg"
    print("Canais posteriores não identificados; usando todos os canais EEG.")
else:
    print(f"Canais posteriores: {posteriores}")

spectrum = raw_ec.compute_psd(
    method="welch",
    fmin=8,
    fmax=12,
    tmin=0,
    tmax=tmax,
    picks=posteriores,
    n_per_seg=int(4 * sfreq_iaf),
    n_overlap=int(2 * sfreq_iaf),
    verbose=False,
)
psd, freqs = spectrum.get_data(return_freqs=True)
potencia_media = psd.mean(axis=0)

iaf_hz = float(freqs[np.argmax(potencia_media)])
periodo_ms = 1000.0 / iaf_hz
L_iaf = int(round(sfreq_iaf / iaf_hz))
WINDOW_MS_IAF = periodo_ms

print(f"Pico alfa individual (IAF): {iaf_hz:.2f} Hz")
print(f"Período alfa: {periodo_ms:.1f} ms")
print(f"Janela sugerida (~1 ciclo): {L_iaf} pts | WINDOW_MS ≈ {WINDOW_MS_IAF:.0f} ms")


tmax usado: 43.38 s | boundaries: 8
Canais posteriores: ['P3', 'Pz', 'P4', 'O1', 'Oz', 'O2', 'PO3', 'POz', 'PO4']
Pico alfa individual (IAF): 10.50 Hz
Período alfa: 95.3 ms
Janela sugerida (~1 ciclo): 24 pts | WINDOW_MS ≈ 95 ms


C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\4222001908.py:4: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\4222001908.py:4: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.



In [18]:
# Espectro médio nos posteriores (8–12 Hz)
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=freqs,
        y=potencia_media,
        mode="lines",
        name="PSD média (posteriores)",
        line=dict(width=2),
    )
)
fig.add_vline(
    x=iaf_hz,
    line_dash="dash",
    line_color="crimson",
    annotation_text=f"IAF = {iaf_hz:.2f} Hz",
)
fig.update_layout(
    title=f"PSD alfa — {file_ec.stem}",
    xaxis_title="Frequência (Hz)",
    yaxis_title="Potência",
    template="plotly_white",
    hovermode="x unified",
    height=400,
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


A PSD dos canais posteriores na condição de olhos fechados indicou pico alfa individual de 10,5 Hz. O período correspondente é de aproximadamente 95,3 ms; portanto, foi adotada uma janela de 24 amostras, equivalente a 96 ms na frequência de amostragem de 250 Hz.

### Método 6 — Validação downstream (EO vs EC)

Subset pequeno (até `N_SUBJECTS_ML` sujeitos com EO e EC). Para cada $L_M$ em `WINDOW_SIZES_ML`, extrai features leves e avalia LogisticRegression com validação cruzada estratificada.

**Atenção:** mais lento que as seções anteriores.


In [19]:
# ==========================
# DOWNSTREAM EO vs EC
# ==========================
from collections import defaultdict

by_subject = defaultdict(dict)
for p in files:
    sid, cond = p.stem.split("_", 1)
    if cond in ("EO", "EC"):
        by_subject[sid][cond] = p

pairs = [
    (sid, paths["EO"], paths["EC"])
    for sid, paths in sorted(by_subject.items())
    if "EO" in paths and "EC" in paths
][:N_SUBJECTS_ML]

print(f"Pares EO/EC usados: {len(pairs)}")
for sid, eo, ec in pairs:
    print(f"  {sid}: {eo.name}, {ec.name}")


Pares EO/EC usados: 3
  sub-010061: sub-010061_EO.set, sub-010061_EC.set
  sub-010062: sub-010062_EO.set, sub-010062_EC.set
  sub-010063: sub-010063_EO.set, sub-010063_EC.set


In [20]:
rows_ml = []

for L_M in tqdm(WINDOW_SIZES_ML, desc="Downstream L_M"):
    max_lag = min(MAX_LAG_CAP, L_M - 1)
    feat_rows = []

    for sid, path_eo, path_ec in pairs:
        for cond, path in (("EO", path_eo), ("EC", path_ec)):
            raw_i = mne.io.read_raw_eeglab(path, preload=True, verbose=False)
            raw_i.pick_types(eeg=True)
            eeg_i = raw_i.get_data()
            sfreq_i = float(raw_i.info["sfreq"])
            eeg_i = eeg_i[:, : int(N_SEC * sfreq_i)]
            motifs_i = trans_motifs(eeg_i)

            thr_i = estimate_threshold(
                motifs_i,
                L_M,
                max_lag,
                n_surrogates=N_SURROGATES,
                percentile=THRESHOLD_PERCENTILE,
                random_state=RANDOM_STATE,
            )["threshold"]

            tvg_i = build_tvg_parallel(
                motifs=motifs_i,
                window_size=L_M,
                max_lag=max_lag,
                threshold=thr_i,
            )
            rea_i = aggregate_static_network(tvg_i)
            edges_t = edges_per_time(tvg_i)
            edges_stats = temporal_statistics(edges_t)
            net = graph_metrics(rea_i)

            feat_rows.append(
                {
                    "subject_id": sid,
                    "condition": cond,
                    "y": 0 if cond == "EO" else 1,
                    "edges_mean": edges_stats["mean"],
                    "edges_std": edges_stats["std"],
                    "global_efficiency": net["global_efficiency"],
                    "mean_degree": float(np.mean(net["degree"])),
                }
            )

    df_feat = pd.DataFrame(feat_rows)
    X = df_feat[["edges_mean", "edges_std", "global_efficiency", "mean_degree"]].values
    y = df_feat["y"].values

    n_splits = min(3, int(np.bincount(y).min()))
    if n_splits < 2:
        acc = float("nan")
        print(f"L_M={L_M}: poucos exemplos por classe — pulando CV")
    else:
        pipe = Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
            ]
        )
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
        scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
        acc = float(np.mean(scores))
        print(f"L_M={L_M:2d} | accuracy CV={acc:.3f} (n={len(y)}, folds={n_splits})")

    rows_ml.append(
        {
            "L_M": L_M,
            "L_M_ms": L_M / sfreq * 1000,
            "accuracy": acc,
            "n_samples": len(y),
        }
    )

df_ml = pd.DataFrame(rows_ml)
if df_ml["accuracy"].notna().any():
    L_ml = int(df_ml.loc[df_ml["accuracy"].idxmax(), "L_M"])
    print(f"\nL_M com maior accuracy: {L_ml} pts (~{L_ml / sfreq * 1000:.1f} ms)")
else:
    L_ml = None
    print("\nNão foi possível estimar accuracy.")
df_ml


Downstream L_M:   0%|          | 0/6 [00:00<?, ?it/s]

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\1477080578.py:9: RuntimeWarning:

Limited 2 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\1477080578.py:9: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.



NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\1477080578.py:9: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\1477080578.py:9: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.



...

C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\1477080578.py:9: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\1477080578.py:9: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.



NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\1477080578.py:9: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\Anne\AppData\Local\Temp\ipykernel_7360\1477080578.py:9: RuntimeWarning:

The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.

Downstream L_M: 100%|██████████| 6/6 [10:15<00:00, 102.65s/it]

L_M=25 | accuracy CV=0.333 (n=6, folds=3)

L_M com maior accuracy: 10 pts (~40.0 ms)


,L_M,L_M_ms,accuracy,n_samples
0,8,32.0,0.500000,6
1,10,40.0,0.833333,6
2,12,48.0,0.166667,6
3,15,60.0,0.333333,6
4,20,80.0,0.500000,6
5,25,100.0,0.333333,6


In [21]:
fig = go.Figure(
    go.Scatter(
        x=df_ml["L_M"],
        y=df_ml["accuracy"],
        mode="lines+markers",
        name="Accuracy EO vs EC",
        marker=dict(size=8, color="darkorange"),
        line=dict(width=2, color="darkorange"),
    )
)
fig.update_layout(
    title="Desempenho downstream (EO vs EC) vs L_M",
    xaxis_title="Tamanho da janela L_M (pts)",
    yaxis_title="Accuracy (CV estratificada)",
    template="plotly_white",
    hovermode="x unified",
    height=450,
    yaxis=dict(range=[0, 1.05]),
)
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


### Resumo — L_M sugerido por método

Consolida Rosário, estabilidade, split-half, faixa fisiológica, **IAF** e downstream.


In [22]:
# ==========================
# RESUMO
# ==========================
sep = (
    (df_na["Na_original"] > 2 * df_na["Na_embaralhado"])
    & ((df_na["Na_original"] - df_na["Na_embaralhado"]) >= 10)
)
if sep.any():
    L_rosario = int(df_na.loc[sep].iloc[0]["L_M"])
    nota_rosario = "menor L_M com separação Na_orig vs Na_shuf"
else:
    L_rosario = None
    nota_rosario = "sem separação clara na faixa"

resumo = [
    {
        "metodo": "1. Rosário (Na orig vs shuf)",
        "L_M_pts": L_rosario,
        "WINDOW_MS": None if L_rosario is None else L_rosario / sfreq * 1000,
        "nota": nota_rosario,
    },
    {
        "metodo": "2. Estabilidade temporal",
        "L_M_pts": L_stab,
        "WINDOW_MS": L_stab / sfreq * 1000,
        "nota": "argmax correlação entre frames consecutivos",
    },
    {
        "metodo": "3. Split-half",
        "L_M_pts": L_split,
        "WINDOW_MS": L_split / sfreq * 1000,
        "nota": "argmax correlação REA metade A vs B",
    },
    {
        "metodo": "4. Fisiológico (faixa)",
        "L_M_pts": f"{L_phys_min}–{L_phys_max}",
        "WINDOW_MS": f"{L_phys_min / sfreq * 1000:.0f}–{L_phys_max / sfreq * 1000:.0f}",
        "nota": "duração ~50–150 ms (alpha/beta)",
    },
    {
        "metodo": "4b. IAF (~1 ciclo alfa)",
        "L_M_pts": L_iaf,
        "WINDOW_MS": WINDOW_MS_IAF,
        "nota": f"pico alfa {iaf_hz:.2f} Hz em {file_ec.name}",
    },
    {
        "metodo": "6. Downstream EO vs EC",
        "L_M_pts": L_ml,
        "WINDOW_MS": None if L_ml is None else L_ml / sfreq * 1000,
        "nota": "argmax accuracy CV (subset pequeno)",
    },
]

df_resumo = pd.DataFrame(resumo)
df_resumo


,metodo,L_M_pts,WINDOW_MS,nota
0,1. Rosário (Na orig vs shuf),8,32.0,menor L_M com separação Na_orig vs Na_shuf
1,2. Estabilidade temporal,25,100.0,argmax correlação entre frames consecutivos
2,3. Split-half,24,96.0,argmax correlação REA metade A vs B
3,4. Fisiológico (faixa),13–25,52–100,duração ~50–150 ms (alpha/beta)
4,4b. IAF (~1 ciclo alfa),24,95.255814,pico alfa 10.50 Hz em sub-010061_EC.set
5,6. Downstream EO vs EC,10,40.0,argmax accuracy CV (subset pequeno)


As análises fisiológica, de reprodutibilidade e de estabilidade temporal convergiram para uma janela entre 96 e 100 ms. Assim, foi adotado o valor de 100 ms (L_M = 25), por ser um parâmetro arredondado, reproduzível e compatível com aproximadamente um ciclo de atividade alfa.

-------

## Determinação do tau máximo